In [1]:
import kymnasium as kym
import gymnasium as gym


env = gym.make(
    'kymnasium/AvoidBlurp-Discrete-Ballistic-Normal-Stage-1',
    render_mode='none'
)
env.reset()

D:\Projects\kymnasium\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


({'mario': array([288., 768., 336., 816.,   0.], dtype=float32),
  'blurps': array([[0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.

In [5]:
RANDOM_SEED = 42

REPLAY_BUFFER_SIZE = 100000
BATCH_SIZE = 32
ALPHA = 0.6
BETA = 1.0

EPSILON_MIN = 0.2
EPSILON_DECAY = 0.9995
INIT_EXPLORATION = 50000
MAX_EPISODES = 10000

INTERVAL_UPDATE = 8
LEARNING_RATE = 0.0003
TAU = 0.005

GAMMA = 0.99995

STATE_DIM = 3 + 30 * 5
STATE_SEQ = 8
ACTION_DIM = 3

N_MONITOR = 100
WIDTH, HEIGHT = 624, 912

In [22]:
from tensorflow import keras
import numpy as np


class ReplayBuffer:
    def __init__(
            self, capacity: int, alpha: float, beta: float
    ):
        self._alpha = alpha
        self._beta = beta
        self._capacity = capacity
        self._states = np.zeros(shape=(self._capacity, STATE_SEQ, STATE_DIM))
        self._actions = np.zeros(shape=(self._capacity, ACTION_DIM))
        self._rewards = np.zeros(shape=(self._capacity,))
        self._next_states = np.zeros(shape=(self._capacity, STATE_SEQ, STATE_DIM))
        self._dones = np.zeros(shape=(self._capacity,))
        self._priorities = np.ones(shape=(self._capacity,))

        self._max_priority = 1.0
        self._size = 0
        self._index = 0

        self._random = np.random.default_rng(RANDOM_SEED)

    @property
    def size_(self):
        return self._size

    def add(self, state, action, reward, next_state, done):
        priority = self._priorities.max() if self._size > 0 else 1.0

        self._states[self._index] = state
        self._actions[self._index] = action
        self._rewards[self._index] = reward
        self._next_states[self._index] = next_state
        self._dones[self._index] = done
        self._priorities[self._index] = priority
        self._index = (self._index + 1) % self._capacity
        self._size = min(self._size + 1, self._capacity)

    def sample(self, batch_size):
        indices = self._random.choice(self._size, size=batch_size, p=prob)

        probs = self._priorities[:self._size] ** self._alpha
        probs /= probs.sum()

        weights = (self._size * probs[indices]) ** self._beta
        weights /= weights.max()
        weights = np.array(weights, dtype=np.float32)

        return (
            self._states[indices],
            self._actions[indices],
            self._rewards[indices],
            self._next_states[indices],
            self._dones[indices],
            indices,
            weights
        )

    def update_priorities(self, indices, priorities):
        for index, priority in zip(indices, priorities):
            self._priorities[index] = priority

In [10]:
from tensorflow import keras


def build_network(input_shape: tuple, n_action: int):
    inputs = keras.Input(
        shape=input_shape
    )
    x = keras.layers.Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(inputs)

    x = keras.layers.Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(x)

    x = keras.layers.Conv1D(
        filters=128,
        kernel_size=2,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(x)
    x = keras.layers.Flatten()(x)

    value = keras.layers.Dense(
        units=128,
        activation='relu',
        kernel_initializer='glorot_normal',
    )(x)
    value = keras.layers.Dense(
        units=1,
        activation='linear',
        kernel_initializer='glorot_normal',
    )(value)

    advantage = keras.layers.Dense(
        units=128,
        activation='relu',
        kernel_initializer='glorot_normal',
    )(x)
    advantage = keras.layers.Dense(
        units=n_action,
        activation='linear',
        kernel_initializer='glorot_normal',
    )(advantage)

    mean_advantage = keras.ops.mean(advantage, axis=1, keepdims=True)
    q_values = value + advantage - mean_advantage

    return keras.models.Model(inputs=inputs, outputs=q_values)

In [ ]:
import kymnasium as kym
from tensorflow import keras
from collections import deque


class Agent(kym.Agent):
    def __init__(
            self,
            behavior_network: keras.models.Model,
            target_network: keras.models.Model,
            state_dim: int,
            state_seq: int,
            action_space: int,
            buffer_size: int,
            alpha: float,
            beta: float,
            gamma: float,
            tau: float,
            epsilon: float,
            epsilon_min: float,
            epsilon_decay: float,
            learning_rate: float,
            batch_size: int,
            n_monitors: int,
            random_state: int
    ):
        self._behavior_network = behavior_network
        self._target_network = target_network
        self._target_network.set_weights(self._behavior_network.get_weights())
        self._state_seq = state_seq
        self._state_dim = state_dim
        self._action_space = action_space
        self._buffer_size = buffer_size
        self._alpha = alpha
        self._beta = beta
        self._gamma = gamma
        self._tau = tau
        self._epsilon = epsilon
        self._epsilon_min = epsilon_min
        self._epsilon_decay = epsilon_decay
        self._learning_rate = learning_rate
        self._batch_size = batch_size
        self._n_monitors = n_monitors
        self._random_state = random_state

        self._replay_buffer = ReplayBuffer(
            capacity=buffer_size,
            alpha=alpha,
            beta=beta
        )
        self._optimizer = keras.optimizers.Adam(learning_rate=self._learning_rate)
        self._objective = keras.losses.Huber(reduction=None)
        self._running_state = deque([
            np.zeros(shape=(self._state_dim, )) for _ in range(self._state_seq)
        ], maxlen=self._state_seq)

        self._losses = deque(maxlen=self._n_monitors)
        self._rewards = deque(maxlen=self._n_monitors)
        self._random = np.random.default_rng(self._random_state)

    @property
    def loss_(self):
        return np.mean(self._losses)

    @property
    def reward_(self):
        return np.mean(self._rewards)

    def to_state(self, obs):
        mario, blurps = obs['mario'], obs['blurps']
        mario = np.array([mario[0] / WIDTH, mario[1] / HEIGHT, mario[-1] / WIDTH])
        blurps = np.array([
            [blurp[0] / WIDTH, blurp[1] / HEIGHT, blurp[4] / WIDTH, blurp[5] / HEIGHT, blurp[6] / HEIGHT] for blurp in blurps
        ])
        blurps = np.ravel(blurps)
        self._running_state.append(np.concatenate([mario, blurps]))
        return keras.ops.expand_dims(self._running_state, axis=0)

    def choose_action(self, state):
        if self._random.random() < self._epsilon:
            return self._random.choice(self._action_space)

        q = self._behavior_network(state)
        q = keras.ops.ravel(q)
        return np.argmax(q)

    def add(self, state, action, reward, next_state, done):
        self._replay_buffer.add(
            state, action, reward, next_state, done
        )

    def train(self, states, actions, rewards, next_states, dones, weights):
        next_actions = keras.ops.argmax(self._behavior_network(next_states), axis=1)
        next_actions = keras.ops.one_hot(next_actions, self._action_space)
        next_action_values = keras.ops.sum(
            self._target_network(next_states) * next_actions, axis=1, keepdims=True
        )
        targets = rewards + (1 - dones) * self._gamma * next_action_values

        with tf.GradientTape() as tape:
            action_values = keras.ops.sum(
                self._behavior_network(states) * actions, axis=1, keepdims=True
            )
            priorities = self._objective(action_values, targets)
            loss = keras.ops.mean(priorities)

        gradients = tape.gradient(loss, self._behavior_network.trainable_variables)
        self._optimizer.apply_gradients(zip(gradients, self._behavior_network.trainable_variables))

        target_weights = self._target_network.get_weights()
        behavior_weights = self._behavior_network.get_weights()

        for i in range(len(target_weights)):
            target_weights[i] = behavior_weights[i] * self._tau + (1 - self._tau) * target_weights[i]
        target_network.set_weights(target_weights)

        return loss, priorities

    def replay(self):
        if self._replay_buffer.size_ < self._batch_size:
            return

        states, actions, rewards, next_states, dones, indices, weights = self._replay_buffer.sample(self._batch_size)

        loss, priorities = self.train(
            states = keras.ops.convert_to_tensor(states),
            actions = keras.ops.convert_to_tensor(actions),
            rewards = keras.ops.convert_to_tensor(rewards),
            next_states = keras.ops.convert_to_tensor(next_states),
            dones = keras.ops.convert_to_tensor(dones),
            weights = keops.convert_to_tensor(weights)
        )
        self._replay_buffer.update_priorities(indices, priorities)
        self._losses.append(loss)

    def track(self, rewards):
        self._rewards.append(rewards)

    def act(self, obs, info):
        state = self.to_state(obs)
        q = self._behavior_network(state)
        q = keras.ops.ravel(q)
        return np.argmax(q)



In [21]:
keras.ops.argmax(behavior_network(keras.ops.ones(shape=(10, 8, 3 + 30 * 5))), axis=1)

<tf.Tensor: shape=(10,), dtype=int32, numpy=array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2], dtype=int32)>

In [ ]:
from tensorflow import keras


def build_network():
    inputs = keras.Input(
        shape=(8, 3 + 30 * 5)
    )
    x = keras.layers.Conv1D(
        filters=32,
        kernel_size=4,
        activation='relu',
        strides=2,
        kernel_initializer='he_normal',
    )(inputs)

    x = keras.layers.Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(x)

    x = keras.layers.Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        strides=1,
        kernel_initializer='he_normal',
    )(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(
        units=512,
        activation='relu',
        kernel_initializer='he_normal',
    )(x)
    value = keras.layers.Dense(
        units=1,
        activation='linear',
        kernel_initializer='glorot_normal',
    )(x)
    advantage = keras.layers.Dense(
        units=3,
        activation='linear',
        kernel_initializer='glorot_normal',
    )(x)

    return keras.models.Model(inputs=inputs, outputs=[value, advantage])


behavior_network = build_network()
target_network = build_network()

n

In [ ]:
from tqdm.auto import tqdm


pbar = tqdm(range(MAX_EPISODES), desc='episode')
reward_tracker = Tracker(N_MONITOR)
loss_tracker = Tracker(N_MONITOR)
epsilon = 1.0

random = np.random.default_rng(RANDOM_SEED)
replay_buffer = ReplayBuffer(REPLAY_BUFFER_SIZE)
max_priority = 1.0

for episode in pbar:
    steps, total_reward = 0.0, 0.0

    done = False
    obs, _ = env.reset()
    obs = preprocess(obs)

    running_states = [
        np.zeros((STATE_DIM, )) for _ in range(STATE_SEQ)
    ]

    del running_states[:1]
    running_states.append(obs)
    state = keras.ops.expand_dims(running_states, axis=0)

    while not done:
        if episode > INIT_EXPLORATION:
            epsilon = max(epsilon * EPSILON_DECAY, EPSILON_MIN)

        if random.random() < epsilon:
            action = choose_action(state).numpy()
        else:
            action = random.choice(ACTION_DIM)

        next_obs, _, terminated, truncated, _ = env.step(action)
        next_obs = preprocess(next_obs)

        del running_states[:1]
        running_states.append(next_obs)
        next_state = keras.ops.expand_dims(running_states, axis=0)

        done = terminated or truncated
        steps += 1
        reward = 0.01 if not done else 0.0
        total_reward += reward

        replay_buffer.push(
            state, keras.ops.one_hot(action, ACTION_DIM), reward, next_state, done, max_priority
        )

        if steps % INTERVAL_UPDATE == 0 and replay_buffer.size_ >= BATCH_SIZE:
            indices, states, actions, rewards, next_states, dones, probs = replay_buffer.sample(BATCH_SIZE)
            loss, priorities = train(states, actions, rewards, next_states, dones, probs)
            replay_buffer.update_priority(indices, priorities)
            max_priority = max(max_priority, np.max(priorities))
            pbar.set_postfix({
              'loss': f'{loss:.5f}'
            })

        state = next_state

    reward_tracker.update(total_reward)
    pbar.set_postfix({
         'reward': f'{reward_tracker.avg_:.5f}'
    })
